# Señal y ruido

Sobreajuste, modelo de ruido gaussiano y verosimilitud

> **Lo que usamos de antes**
>
> De **?@def-problema**, los predictores $\mathbf{x}_i$, la respuesta $y_i$ y el símbolo $\approx$. De **?@def-hipotesis**, la clase de hipótesis. Del anexo de probabilidad, densidad, esperanza y varianza.

## Planteamiento

El capítulo 1 dejó dos cabos sueltos, y los dos eran observaciones sin explicación.

El primero: un árbol sin límite de profundidad se equivocaba en medio euro por anuncio sobre los datos con los que se construyó, y en 78 euros sobre los apartados. Vimos lo mismo dibujado en **?@fig-trampa**, con un polinomio que pasaba por los 12 puntos. Lo que no sabemos todavía es si aquello fue mala suerte de esos datos concretos o algo inevitable.

El segundo: el símbolo $\approx$ de **?@def-problema**. Dijimos que los puntos no están sobre la curva sino alrededor, pero no dijimos cuánto ni cómo.

Este capítulo cierra los dos, y en el orden contrario al que se plantearon.

Primero demostramos que lo del árbol **siempre** ocurre: con clases de hipótesis suficientemente ricas, ajustar los datos a la perfección es posible en cualquier conjunto de datos, de modo que conseguirlo no informa de nada (<a href="#thm-interpolacion" class="quarto-xref">Teorema 1</a>). No hace falta un árbol ni un polinomio: le pasa a cualquier modelo con libertad suficiente.

Después convertimos el $\approx$ en un modelo concreto, $y = f(\mathbf{x}) + \varepsilon$ con $\varepsilon$ aleatorio. En cuanto se supone algo sobre $\varepsilon$, elegir un modelo deja de ser una cuestión de criterio y se convierte en un problema de **verosimilitud**, que es un problema que sabemos resolver. Ese paso es el que fija la pérdida, y con ello la segunda de las tres decisiones de **?@sec-tres-decisiones**.

## Datos, señal y ruido

Recordemos la condición que da contenido a **?@def-problema**: el error se mide sobre observaciones **no vistas**, es decir, distintas de las que se han usado para ajustar. Sin esa condición el problema es trivial, y el primer resultado del capítulo dice en qué sentido exacto lo es.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

rng = np.random.default_rng(42)
kw_puntos = dict(color="black", facecolors="none", s=40, alpha=0.6, label="datos")

In [ ]:
# TODO: completar en clase

Figura 1: Descomposición de un conjunto de datos (puntos) en una señal subyacente (línea discontinua) y ruido (segmentos naranjas).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(x, y, **kw_puntos)
ax.plot(x, senal, color="black", linestyle="--",
        label=r"Señal: $f(x_i) = 2x_i + 1$")
for i in range(n):
    etiqueta = r"Ruido: $\epsilon_i$" if i == 0 else None
    ax.plot([x[i]] * 2, [senal[i], y[i]],
            color="#ff5700", alpha=0.5, linewidth=0.9, label=etiqueta)
ax.set(xlabel="$x$", ylabel="$y$")
ax.legend()
plt.show()

<span class="theorem-title">**Definición 1 (Modelo señal + ruido)**</span> $$
y_i= f(\mathbf{x}_i) + \varepsilon_i, \qquad \varepsilon_i \text{ iid}, \quad \mathbb{E}\!\left[ \varepsilon_i \right] = 0.
 \qquad(1)$$

A $f$ la llamamos **señal** y a $\varepsilon_i$ **ruido**.

Lo que queremos aprender es $f$, **no** los valores $y_i$.

<span class="theorem-title">**Definición 2 (Error de entrenamiento y error de test)**</span> Es **?@def-perdida** con pérdida cuadrática, restringida a un subconjunto. Dado un conjunto $\mathcal{S}$ y un modelo ajustado $\hat{f}$, $$
\hat{R}_{\mathcal{S}}(\hat{f}) = \frac{1}{\left\lvert \mathcal{S} \right\rvert} \sum_{i \in \mathcal{S}} \left( y_i- \hat{f}(\mathbf{x}_i) \right)^2 .
$$

> **Note**
>
> De momento el cuadrado es una elección arbitraria: lo estamos **midiendo**, no justificando. En <a href="#sec-mle-teorema" class="quarto-xref">Sección 5</a> demostraremos que se deduce de suponer ruido gaussiano.

## Interpolar no es aprender

<span class="theorem-title">**Definición 3 (Sobreajuste)**</span> Se produce **sobreajuste** cuando aumentar la complejidad del modelo mejora su error de entrenamiento y **empeora** su error de test.

<a href="#def-sobreajuste" class="quarto-xref">Definición 3</a> es más exigente que la descripción del capítulo 1. Allí dijimos que el error de entrenamiento era mucho menor que el de test, lo que puede ocurrir por casualidad con una muestra pequeña. Aquí pedimos que la diferencia responda a un cambio en la complejidad del modelo, que es lo que permite hacer algo al respecto: si moverla en un sentido empeora el test, se mueve en el otro.

El siguiente resultado no suele aparecer en los cursos introductorios.

<span class="theorem-title">**Teorema 1 (Interpolación exacta)**</span> Sean $x_1, \dots, x_{n} \in \mathbb{R}$ **distintos dos a dos** y sean $y_1, \dots, y_{n} \in \mathbb{R}$ cualesquiera. Entonces existe un polinomio $P$ de grado $\leq n- 1$ tal que $P(x_i) = y_i$ para todo $i$; es decir, con error de entrenamiento **exactamente cero**.

<span class="proof-title">*Demostración*. </span>Construimos la solución. Para cada $j$ defínase la *base de Lagrange* $$
\ell_j(x) = \prod_{k \neq j} \frac{x - x_k}{x_j - x_k},
$$ que está bien definida porque los $x_k$ son distintos. Evaluando, si $i = j$ todos los factores valen $1$, y si $i \neq j$ el factor $k = i$ se anula; por tanto $\ell_j(x_i) = \delta_{ij}$. Tomando $$
P(x) = \sum_{j=1}^{n} y_j\, \ell_j(x)
$$ se obtiene $P(x_i) = \sum_j y_j \delta_{ij} = y_i$. Cada $\ell_j$ tiene grado $n- 1$, luego $P$ también.

> **Consecuencia**
>
> Un error de entrenamiento de cero **no es evidencia de nada**: siempre se puede conseguir. Por tanto, una afirmación del tipo *“nuestro modelo acierta el 100 %”* no informa de nada mientras no se diga sobre qué datos se ha medido.

Lo comprobamos numéricamente. Interpolamos los 20 puntos y después generamos datos nuevos con la **misma señal y ruido distinto**.

Figura 2: El mismo interpolador: error nulo sobre los datos de entrenamiento (izquierda) y error grande sobre datos nuevos (derecha).

In [ ]:
from scipy import interpolate

f_int = interpolate.interp1d(x, y, kind="cubic")
x_denso = np.linspace(0, 10, 200)
y_int = f_int(x_denso)

fig, ax = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
ax[0].scatter(x, y, **kw_puntos)
ax[0].plot(x_denso, y_int, color="#ff5700", linestyle="--", label="interpolador")
ax[0].set(title="Datos de entrenamiento", xlabel="$x$", ylabel="$y$")
ax[0].legend()

y_nuevo = 2.0 * x_denso + 1.0 + rng.normal(scale=3.0, size=200)
ax[1].scatter(x_denso, y_nuevo, **kw_puntos)
ax[1].plot(x_denso, y_int, color="#ff5700", linestyle="--", label="interpolador")
ax[1].set(title="Datos nuevos, misma señal", xlabel="$x$")
ax[1].legend()
plt.show()

## Modelizar el ruido

Si queremos aprender $f$ en <a href="#eq-senal-ruido" class="quarto-xref">Ecuación 1</a>, necesitamos poder hablar matemáticamente de $\varepsilon_i$. Supondremos que el ruido es gaussiano.

<span class="theorem-title">**Definición 4 (Densidad normal)**</span> $$
p(x; \mu, \sigma) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left( -\frac{(x-\mu)^2}{2\sigma^2} \right).
 \qquad(2)$$

> **De Estadística**
>
> Usaremos cuatro propiedades de la normal: la constante de normalización, y su comportamiento bajo traslación (**?@thm-traslacion**), bajo cambio de escala (**?@thm-escala**) y la estandarización que se sigue de ambas (**?@cor-estandarizacion**), más $\mathbb{E}\!\left[ \varepsilon \right]=\mu$ y $\mathrm{Var}\!\left( \varepsilon \right)=\sigma^2$ (**?@thm-momentos**). Están enunciadas y demostradas en el [repaso de probabilidad](../curso/probabilidad.qmd). Aquí solo necesitamos la primera de ellas.

<span class="theorem-title">**Corolario 1 (Del modelo de ruido a la distribución del objetivo)**</span> $\varepsilon_i \sim \mathcal{N}(0, \sigma^2)$ si y solo si $y_i\,\vert\,\mathbf{x}_i \sim \mathcal{N}(f(\mathbf{x}_i), \sigma^2)$.

<span class="proof-title">*Demostración*. </span>**?@thm-traslacion** con $c = f(\mathbf{x}_i)$.

El corolario convierte una hipótesis sobre los **errores** en una distribución para los **datos**. A una distribución se le puede calcular la verosimilitud, y eso es lo que haremos a continuación.

## Verosimilitud

<span class="theorem-title">**Definición 5 (Modelo lineal-gaussiano)**</span> Con $p= 1$: $y_i\sim \mathcal{N}(w_0+ w_1 x_i,\ \sigma^2)$, de forma independiente.

> **Sobre la independencia**
>
> *Independientes* significa aquí condicionalmente a las $\mathbf{x}_i$. Es una hipótesis, no una verdad. En la **?@sec-grupos** del capítulo 8 veremos qué ocurre cuando no se cumple: un mismo anfitrión con doce anuncios en Madrid la rompe.

<span class="theorem-title">**Definición 6 (Verosimilitud y log-verosimilitud)**</span> $$
L(\boldsymbol{w}) = \prod_{i=1}^{n} p\left(y_i;\ f(\mathbf{x}_i),\ \sigma\right),
\qquad
\ell(\boldsymbol{w}) = \sum_{i=1}^{n} \log p\left(y_i;\ f(\mathbf{x}_i),\ \sigma\right).
$$

<span class="theorem-title">**Lema 1 (El logaritmo preserva el argmax)**</span> Si $g$ es estrictamente creciente, $\mathop{\mathrm{arg\,max}}_{\boldsymbol{w}} g(L(\boldsymbol{w})) = \mathop{\mathrm{arg\,max}}_{\boldsymbol{w}} L(\boldsymbol{w})$. En particular $\mathop{\mathrm{arg\,max}}L= \mathop{\mathrm{arg\,max}}\ell= \mathop{\mathrm{arg\,min}}(-\ell)$.

<span class="proof-title">*Demostración*. </span>Si $\boldsymbol{w}^{\star}$ maximiza $L$ entonces $L(\boldsymbol{w}) \leq L(\boldsymbol{w}^\star)$ para todo $\boldsymbol{w}$, y aplicando $g$ creciente, $g(L(\boldsymbol{w})) \leq g(L(\boldsymbol{w}^\star))$. El recíproco es idéntico usando que $g$ es estrictamente creciente.

<span class="theorem-title">**Definición 7 (Estimación máximo-verosímil)**</span> $\hat{\boldsymbol{w}}= \mathop{\mathrm{arg\,max}}_{\boldsymbol{w}} L(\boldsymbol{w}) = \mathop{\mathrm{arg\,min}}_{\boldsymbol{w}} \left(-\ell(\boldsymbol{w})\right)$.

> **Convenio: siempre minimizamos**
>
> Aunque el problema nazca como una maximización, en este curso **siempre** lo escribiremos como minimización. Es el convenio de toda la literatura de optimización y evita errores de signo en el capítulo 5.

### Resolución por búsqueda en rejilla

Empezamos por una búsqueda exhaustiva en una rejilla, para evaluar su coste computacional.

In [ ]:
# TODO: completar en clase

In [ ]:
rejilla_w0 = np.linspace(-5, 5, 60)
rejilla_w1 = np.linspace(-1, 3, 60)
L = np.array([[log_verosimilitud(a, b, x, y) for b in rejilla_w1] for a in rejilla_w0])

fig, ax = plt.subplots(figsize=(5.5, 4))
cs = ax.contourf(rejilla_w1, rejilla_w0, L, levels=30)
i, j = np.unravel_index(L.argmax(), L.shape)
ax.scatter([rejilla_w1[j]], [rejilla_w0[i]], color="#ff5700", s=60, label="máximo en la rejilla")
ax.scatter([2.0], [1.0], color="white", marker="x", s=60, label="verdad")
ax.set(xlabel="$w_1$", ylabel="$w_0$")
ax.legend()
plt.colorbar(cs, ax=ax, label=r"$\ell(w)$")
plt.show()

Con dos parámetros y 60 valores cada uno son 3.600 evaluaciones, y en el capítulo 1 ya vimos que con $p= 20$ predictores serían $60^{21}$. **Hace falta cálculo**, y eso es el capítulo 5.

## Error frecuente

**“Mi $R^2$ de entrenamiento es 0,999”**

Es exactamente <a href="#thm-interpolacion" class="quarto-xref">Teorema 1</a> en acción. Añade grados de libertad suficientes y cualquiera consigue ese número. La pregunta correcta no es *cuánto ajusta* sino *sobre qué datos se ha medido*. Volveremos sobre esto con el aparato completo en el capítulo 8.

## Notación ↔ código

| Matemáticas       | Código            | Nota                    |
|-------------------|-------------------|-------------------------|
| $n$               | `n`, `X.shape[0]` | número de observaciones |
| $p$               | `p`, `X.shape[1]` | número de variables     |
| $\boldsymbol{w}$  | `w`               | vector de coeficientes  |
| $\sigma$          | `sigma`           | escala del ruido        |
| $f(\mathbf{x}_i)$ | `f(x)`            | la señal                |
| $\varepsilon_i$   | `ruido`           | lo que no explicamos    |

## Ejercicios

<span class="theorem-title">**Ejercicio 1**</span> Sean los puntos $(1, 3)$, $(2, 5)$, $(4, 2)$. Construye explícitamente el polinomio interpolador de Lagrange y comprueba que pasa por los tres puntos.

<span class="theorem-title">**Ejercicio 2**</span> Tres titulares de prensa afirman: (a) “el modelo predice el impago con un 99 % de acierto”; (b) “hemos reducido el error a cero”; (c) “validado sobre 10 millones de registros”. Para cada uno, di qué información falta para poder juzgarlo.

<span class="theorem-title">**Ejercicio 3**</span> Supón $\varepsilon_i \sim \mathcal{N}(0, \sigma^2)$. Escribe la log-verosimilitud de una única observación y dibuja, sin calcular nada, cómo cambia al alejarse $y_i$ de $f(\mathbf{x}_i)$.

## Resumen

- Error de entrenamiento cero se consigue **siempre** (<a href="#thm-interpolacion" class="quarto-xref">Teorema 1</a>) y por tanto no es evidencia de nada.
- Suponer una distribución para el ruido convierte el problema en uno de **verosimilitud** (<a href="#cor-target" class="quarto-xref">Corolario 1</a>).
- Maximizar la verosimilitud por fuerza bruta no escala. Necesitamos gradientes.